# Attention med synlige tall

**Du skal kunne:** regne ut attention, tolke en rad i et attention-kart og forklare
hvorfor en språkmodell trenger en maske. Vi bruker tre små, håndlagde vektorer.
Denne notatboken trener ingenting; i neste notatbok lærer modellen selv slike
representasjoner fra data.

En analogi: En deltaker har et spørsmål (**query**), hvert oppslagskort har en
etikett (**key**) og et innhold (**value**). Likheten mellom spørsmål og etiketter
bestemmer hvor mye innhold vi blander fra hvert kort. Analogien forklarer
regnemåten; tallene er ikke menneskelige spørsmål eller forståelse.


## Gjør arbeidsbenken klar

`import` henter verktøy; `as` gir et kortere navn. `Path` håndterer filstier.
Vi finner kursmappen og legger `startkode` i Pythons søkesti, slik at samme kode
virker fra VS Code og JupyterLab. `sett_seed(42)` gir et fast startpunkt for
slumptall; det gjør sammenligning lettere, men lover ikke identiske tall på ulike
maskiner. `velg_enhet()` velger NVIDIA-GPU når CUDA er tilgjengelig, ellers CPU.
En **enhet** er stedet beregningene skjer. Modell og data må ligge på samme enhet.

**Før kjøring:** Gjett om utskriften blir `cpu` eller `cuda`.
**Etter kjøring:** Kontroller at det stemmer med installasjonssjekken.


In [ ]:
from pathlib import Path
import sys
import torch
import matplotlib.pyplot as plt

# Finn kursmappen både fra en notatbok og fra prosjektroten.
her = Path.cwd().resolve()
kandidater = [her, *her.parents, her / "05-dypere-maskinlaering"]
kursmappe = next(p for p in kandidater if (p / "startkode" / "kursverktoy.py").exists())
if str(kursmappe / "startkode") not in sys.path:
    sys.path.insert(0, str(kursmappe / "startkode"))
from kursverktoy import sett_seed, velg_enhet
sett_seed(42)
enhet = velg_enhet()
print("PyTorch:", torch.__version__, "| beregner på:", enhet)


## Fra query og key til vekter

`Q @ K.T` sammenligner hver query med hver key ved et prikkprodukt. `.T` bytter
rader og kolonner. Vi deler på kvadratroten av vektorbredden for å dempe veksten
i skårene når vektorene blir bredere. `softmax(dim=-1)` gjør hver rad til positive
vekter som summerer til én. `@ V` lager en vektet blanding av innholdet.

Vi bruker etikettene «jeg», «ser», «bilen» for å kunne lese aksene. Tallene er valgt
for hånd og har ingen lært språklig betydning.
**Forutsi:** Query for «jeg» er `[1, 0]`. Hvilken key ligner mest?


In [ ]:
ordene = ["jeg", "ser", "bilen"]
Q = torch.tensor([[1., 0.], [0., 1.], [1., 1.]])
K = torch.tensor([[1., 0.], [0., 1.], [1., 1.]])
V = torch.tensor([[10., 0.], [0., 10.], [5., 5.]])
skaar = Q @ K.T / (Q.shape[-1] ** 0.5)
vekter = skaar.softmax(dim=-1)
blanding = vekter @ V
print("Skårer:\n", skaar)
print("Attention-vekter:\n", vekter)
print("Radsummer:", vekter.sum(-1))
print("Blandet innhold:\n", blanding)


**Kontroll:** Første query gir like skårer til «jeg» og «bilen». Det er prikkproduktet
som styrer, ikke bare om vektorene er helt like. Alle tre får positiv vekt før maskering.

## Skjul framtiden

Når modellen skal gjette neste tegn, må den ikke se fasiten som ligger senere i
treningssekvensen. `triu(diagonal=1)` markerer alt over diagonalen: posisjonene
som ligger i framtiden. `masked_fill` setter disse skårene til minus uendelig.
Softmax gir dem da vekt null. Den nåværende posisjonen får fortsatt se seg selv,
fordi målet ligger **ett tegn etter** den posisjonen.


In [ ]:
maske = torch.triu(torch.ones(3, 3, dtype=torch.bool), diagonal=1)
kausale_vekter = skaar.masked_fill(maske, float("-inf")).softmax(dim=-1)
print(kausale_vekter)
fig, akser = plt.subplots(1, 2, figsize=(9, 4))
for akse, matrise, tittel in zip(akser, [vekter, kausale_vekter],
                                ["Uten maske", "Med kausal maske"]):
    bilde = akse.imshow(matrise, vmin=0, vmax=1, cmap="Blues")
    akse.set_xticks(range(3), ordene)
    akse.set_yticks(range(3), ordene)
    akse.set_xlabel("Henter fra key")
    akse.set_ylabel("Query")
    akse.set_title(tittel)
fig.colorbar(bilde, ax=akser, label="Vekt", shrink=0.8)
plt.show()


## Slik lærer et attention-lag representasjonene

I en modell lager tre lineære lag Q, K og V fra inputrepresentasjonen X. Her får
vektene tilfeldige startverdier, så ingenting er trent.
`bias=False` utelater startverdien i de lineære lagene for å gjøre eksempelet lite.
`requires_grad` viser at vektene kan motta gradienter gjennom attention-regnestykket.
Neste notatbok bruker dette prinsippet med flere attention-hoder samtidig.


In [ ]:
from torch import nn
X = torch.tensor([[1., 0., 0., 0.], [0., 1., 0., 0.], [0., 0., 1., 0.]])
lag_q, lag_k, lag_v = (nn.Linear(4, 4, bias=False) for _ in range(3))
q, k, v = lag_q(X), lag_k(X), lag_v(X)
laerte_vekter = ((q @ k.T / 2).masked_fill(maske, float("-inf"))).softmax(-1)
output = laerte_vekter @ v
print("Output-form:", output.shape, "kan få gradienter:", lag_q.weight.requires_grad)


## Eksperiment og grense for tolkning

1. Bytt første query til `[0, 3]`. Hvilke nøkler får mer vekt?
2. Endre V uten å endre Q eller K. Attention-vektene er uendret, men output endres.
3. Forklar hvorfor første rad i den kausale matrisen må være `[1, 0, 0]`.

Et lyst felt viser hvor mye en representasjon vektes i dette laget og hodet.
Det er ikke alene en forklaring på hvorfor modellen svarte noe, og sier ikke
at den har forstått ordene.

Kilde: [Attention Is All You Need](https://arxiv.org/abs/1706.03762).
**Neste:** [07-tren-en-liten-sprakmodell.ipynb](07-tren-en-liten-sprakmodell.ipynb).
